# ConvNeXt V2 — Local CPU Test

Automatically finds the latest trained checkpoint on **Google Drive** (via Drive for Desktop -- no manual download) and evaluates it against the CheXpert validation set entirely on your local machine -- CPU only, no Colab, no compute units spent.

**Before running:**
1. Make sure [Google Drive for Desktop](https://www.google.com/drive/download/) is installed and signed in to the same account used for training -- this notebook reads checkpoints directly from `G:\My Drive\convnextv2_chexpert_checkpoints\`, no manual download needed.
2. Run all cells in order -- the config cell automatically finds the most recent checkpoint on Drive and detects `NUM_CLASSES` from its experiment folder name. Override `CHECKPOINT_INDEX` or `NUM_CLASSES` directly in that cell if you want a different checkpoint or auto-detection got it wrong.

This reuses the same model-building code, dataloader, and metric functions (`meanAUC`/`meanF1`/etc.) as the Colab training notebooks -- just run directly, without the training loop, checkpoint-saving-on-improvement logic, or any of the Colab-specific setup (Drive mount, Kaggle download). `A:\Dataset\archive` is used as the local CheXpert copy already on this machine.

**Note on speed**: CPU inference on ~234 validation images is slow for larger models -- expect anywhere from under a minute (DenseNet-121) to several minutes (Swin-L/RAD-DINO) depending on model size. `TEST_AUGMENT=False` (single center-crop) is the default here for speed; the Colab notebooks used 10-crop test-time augmentation (`TEST_AUGMENT=True`) for their reported numbers, which is ~10x slower but slightly more accurate -- flip it on if you want numbers directly comparable to the training run's reported metrics, off for a quick sanity check.

In [ ]:
import sys, os
sys.path.insert(0, r"C:\Users\micha\Desktop\Coding\Project\Repos\ark_plus_repo\Ark_Plus\Finetuning")
import torch
from torch.utils.data import DataLoader
from dataloader import CheXpert, build_transform_classification
from models import build_classification_model
from utils import meanAUC, meanMCC, meanAP, meanF1, meanRecall, meanAccuracy, meanECE

# ---- Auto-discover the checkpoint from Google Drive (Drive for Desktop mount) -- no manual
# download needed. Picks the most recently modified checkpoint for this model by default.
import glob, re

DRIVE_CHECKPOINT_DIR = r"G:\My Drive\convnextv2_chexpert_checkpoints"
CHECKPOINT_INDEX = -1     # -1 = most recently modified; set to a specific index from the printed list to pick a different one

_available_ckpts = sorted(
    glob.glob(os.path.join(DRIVE_CHECKPOINT_DIR, "**", "*.pth.tar"), recursive=True),
    key=os.path.getmtime,
)
if not _available_ckpts:
    raise FileNotFoundError(
        f"No checkpoints found under {DRIVE_CHECKPOINT_DIR} -- make sure training has run in "
        "Colab and Google Drive for Desktop has finished syncing."
    )
print(f"Found {len(_available_ckpts)} checkpoint(s) on Drive (oldest to newest):")
for _i, _p in enumerate(_available_ckpts):
    print(f"  [{_i}] {_p}")

CHECKPOINT_PATH = _available_ckpts[CHECKPOINT_INDEX]
print(f"\nUsing: {CHECKPOINT_PATH}")

# NUM_CLASSES is auto-detected from the checkpoint's experiment folder name (e.g. "...14class...")
_match = re.search(r"(\d+)class", CHECKPOINT_PATH)
NUM_CLASSES = int(_match.group(1)) if _match else 14  # <-- override here if auto-detection looks wrong
print(f"Detected NUM_CLASSES = {NUM_CLASSES} (override this line if that's wrong)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # uses your local GPU if available -- this is
# local hardware, not Colab, so there's no compute-unit cost either way; GPU is just faster.
print(f"Using device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))

BATCH_SIZE = 8            # bump this up if using a GPU -- plenty of headroom over CPU
FEW_SHOT = -1             # -1 = use all 234 validation images; set to a small number (e.g. 1) for a quick single-image test
TEST_AUGMENT = False       # False = fast single-crop check; True = 10-crop, matches training-reported numbers (slower)
DATA_DIR = r"A:\Dataset\archive"  # local CheXpert dataset root

# Model architecture config -- matches this notebook's corresponding Colab training notebook.
MODEL_NAME = "convnextv2_base"
INIT = "convnextv2"
INPUT_SIZE = 224
IMG_SIZE = 256
NORMALIZATION = "imagenet"

In [ ]:
diseases_14 = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
               'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax',
               'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']
competition_5 = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']
class_indices_5 = [13, 7, 11, 10, 15]  # raw CSV column indices for competition_5, in that order

if NUM_CLASSES == 14:
    diseases = diseases_14
    ds_class_indices = None
    test_diseases = [diseases_14.index(c) for c in competition_5]
else:
    diseases = competition_5
    ds_class_indices = class_indices_5
    test_diseases = None

class _Args:
    pass
args = _Args()
args.model_name = MODEL_NAME
args.init = INIT
args.num_class = NUM_CLASSES
args.input_size = INPUT_SIZE  # needed directly by some architectures (e.g. Swin-L/Swin-B via SwinTransformer)
args.pretrained_weights = None  # build the base architecture; overwritten by the checkpoint right after

print("Building model architecture (may download the base pretrained weights once if not already cached)...")
model = build_classification_model(args)
model = model.to(DEVICE)

print("Loading checkpoint:", CHECKPOINT_PATH)
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state_dict = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
state_dict = {(k[len("module."):] if k.startswith("module.") else k): v for k, v in state_dict.items()}
missing, unexpected = model.load_state_dict(state_dict, strict=False)
if missing:
    print("WARNING -- missing keys (checkpoint may not match this model config):", missing)
if unexpected:
    print("WARNING -- unexpected keys (checkpoint may not match this model config):", unexpected)
model.eval()
print("Checkpoint loaded OK.")

In [ ]:
transform = build_transform_classification(normalize=NORMALIZATION, mode="test", crop_size=INPUT_SIZE, resize=IMG_SIZE, test_augment=TEST_AUGMENT)
dataset_val = CheXpert(images_path=DATA_DIR, file_path=os.path.join(DATA_DIR, "valid.csv"),
                        augment=transform, class_indices=ds_class_indices, few_shot=FEW_SHOT)
loader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Validation set: {len(dataset_val)} images, {len(loader)} batches at batch_size={BATCH_SIZE}")

In [ ]:
print(f"Running inference on {DEVICE.upper()}" + (" -- this may take a while for larger models..." if DEVICE == "cpu" else "..."))
y_all = torch.FloatTensor()
p_all = torch.FloatTensor()
with torch.no_grad():
    for i, (samples, targets) in enumerate(loader):
        y_all = torch.cat((y_all, targets), 0)
        if samples.dim() == 4:
            bs, c, h, w = samples.shape
            n_crops = 1
        else:
            bs, n_crops, c, h, w = samples.shape
        inp = samples.view(-1, c, h, w).to(DEVICE)
        out = model(inp)
        out = torch.sigmoid(out).cpu()
        out_mean = out.view(bs, n_crops, -1).mean(1)
        p_all = torch.cat((p_all, out_mean), 0)
        if (i + 1) % 5 == 0 or (i + 1) == len(loader):
            print(f"  batch {i + 1}/{len(loader)}")

y_np = y_all.numpy()
p_np = p_all.numpy()
if test_diseases is not None:
    y_np = y_np[:, test_diseases]
    p_np = p_np[:, test_diseases]
print("Inference done.")

In [ ]:
import pandas as pd

mAUC, auc_scores = meanAUC(y_np, p_np)
mMCC, mcc_scores = meanMCC(y_np, p_np)
mAP, ap_scores = meanAP(y_np, p_np)
mF1, f1_scores = meanF1(y_np, p_np)
mRecall, recall_scores = meanRecall(y_np, p_np)
mAccuracy, accuracy_scores = meanAccuracy(y_np, p_np)
mECE, ece_scores = meanECE(y_np, p_np)

# Raw per-image predicted probability vs. ground truth -- always shown, since this is
# the only meaningful output when testing on very few images (aggregate metrics like
# AUC/F1 need both positive and negative examples per class to mean anything, so they
# come out empty/NaN for classes with no positive case in a tiny sample).
print(f"Showing raw predictions for all {len(dataset_val)} tested image(s):\n")
for idx in range(len(dataset_val)):
    print("Image:", dataset_val.img_list[idx])
    for d, pred, true in zip(competition_5, p_np[idx], y_np[idx]):
        print(f"    {d:<20s} predicted={pred:.4f}   ground_truth={true:.0f}")
    print()

if len(dataset_val) < 10:
    print("Note: aggregate metrics are not meaningful with this few images -- "
          "treat the per-image predictions above as the real result.\n")

# meanAUC/meanMCC/etc. silently drop any class with zero positive examples in this sample
# from their per-class list (without saying which class was dropped) -- with very few images
# that means the returned lists can come back shorter than competition_5, or empty, and can't
# be reliably lined back up with specific disease names. Only show the per-class table when
# every class actually got scored; otherwise just report the (likely NaN) means.
score_lists = {"AUC": auc_scores, "MCC": mcc_scores, "AP": ap_scores,
               "F1": f1_scores, "Recall": recall_scores, "Accuracy": accuracy_scores,
               "ECE": ece_scores}
if all(len(v) == len(competition_5) for v in score_lists.values()):
    df = pd.DataFrame({"Disease": competition_5, **score_lists}).set_index("Disease").round(4)
    display(df)
else:
    print("Skipping the per-class table -- at least one class had zero positive examples in "
          "this sample, so per-class scores can't be reliably matched back to disease names.")

print(f"\nMean: AUC={mAUC:.4f}  MCC={mMCC:.4f}  AP={mAP:.4f}  F1={mF1:.4f}  Recall={mRecall:.4f}  Accuracy={mAccuracy:.4f}  ECE={mECE:.4f}")

## Performance charts
ROC curves, precision-recall curves, and a bar chart of the mean metrics -- the visual version of the table above. Needs the validation-set inference cells above to have actually run (not the single-image section).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, name in enumerate(competition_5):
    if not np.any(y_np[:, i]):
        continue  # ROC/PR are undefined for a class with no positive examples in this sample
    fpr, tpr, _ = roc_curve(y_np[:, i], p_np[:, i])
    axes[0].plot(fpr, tpr, label=name)
    prec, rec, _ = precision_recall_curve(y_np[:, i], p_np[:, i])
    axes[1].plot(rec, prec, label=name)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curves"); axes[0].legend(fontsize=8)
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curves"); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

fig2, ax = plt.subplots(figsize=(6, 4))
metrics_bar = {"AUC": mAUC, "MCC": mMCC, "AP": mAP, "F1": mF1, "Recall": mRecall, "Accuracy": mAccuracy}
ax.bar(list(metrics_bar.keys()), list(metrics_bar.values()))
ax.set_ylim(min(0, min(metrics_bar.values())), 1)
ax.set_title("Mean metrics across all classes")
plt.tight_layout()
plt.show()

## Test on your own image
Everything above tests against the CheXpert validation set (real chest X-rays with known labels, used to compute AUC/F1/etc.). This section instead runs the model on **any single image file you point it at** -- doesn't need to be part of CheXpert at all, and there's no ground truth to compare against, just the model's raw predictions.

Reuses the exact same `model` and `transform` built above, so this only works after running all the cells above first.

In [ ]:
from PIL import Image

_run_single_image = input("Test on a single image? [y/N]: ").strip().lower().startswith("y")

if _run_single_image:
    YOUR_IMAGE_PATH = input("Enter the image path (jpg/png): ").strip().strip('"')

    img = Image.open(YOUR_IMAGE_PATH).convert('RGB')
    img_tensor = transform(img)
    img_tensor = img_tensor.unsqueeze(0)  # add a batch dimension of 1

    with torch.no_grad():
        if img_tensor.dim() == 5:
            # TenCrop mode (TEST_AUGMENT=True): (1, n_crops, C, H, W)
            bs, n_crops, c, h, w = img_tensor.shape
        else:
            # single-crop mode (TEST_AUGMENT=False): (1, C, H, W)
            bs, c, h, w = img_tensor.shape
            n_crops = 1
        inp = img_tensor.view(-1, c, h, w).to(DEVICE)
        out = model(inp)
        out = torch.sigmoid(out).cpu()
        pred = out.view(bs, n_crops, -1).mean(1).squeeze(0)

    print(f"Predictions for: {YOUR_IMAGE_PATH}\n")
    for d, p in zip(diseases, pred.tolist()):
        print(f"  {d:<28s} predicted={p:.4f}")
else:
    print("Skipped -- no image selected. Re-run this cell if you want to test one.")


## Grad-CAM: where is the model looking?
Highlights which regions of the image most influenced the model's prediction for a chosen disease -- computed by backpropagating that disease's logit to an internal feature layer and weighting the layer's activations by how much they affected the prediction. Red/warm regions contributed most; blue/cool regions contributed least.

Requires the "Test on your own image" cell above to have run first (reuses `model`, `img`, `img_tensor`).

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt

if not _run_single_image:
    print("Skipped -- no image was selected in the \"Test on your own image\" cell above, "
          "so there's nothing to run Grad-CAM on. Re-run that cell and answer 'y' if you want a visualization.")
else:
    class GradCAM:
        def __init__(self, model, target_layer, reshape_fn):
            self.model = model
            self.activations = None
            self.gradients = None
            self.reshape_fn = reshape_fn
            target_layer.register_forward_hook(self._save_activation)
            target_layer.register_full_backward_hook(self._save_gradient)

        def _save_activation(self, module, inp, out):
            self.activations = (out[0] if isinstance(out, tuple) else out).detach()

        def _save_gradient(self, module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        def __call__(self, x, class_idx):
            self.model.zero_grad()
            x = x.clone().to(next(self.model.parameters()).device).requires_grad_(True)
            out = self.model(x)
            score = out[:, class_idx].sum()
            score.backward()
            act = self.reshape_fn(self.activations[0])
            grad = self.reshape_fn(self.gradients[0])
            weights = grad.mean(dim=(1, 2))
            cam = torch.einsum("c,chw->hw", weights, act)
            cam = torch.relu(cam)
            cam = cam - cam.min()
            if cam.max() > 0:
                cam = cam / cam.max()
            return cam.detach().cpu(), out.detach().cpu()

    RESHAPE_FN = lambda t: t  # already spatial (C, H, W), no reshape needed
    TARGET_LAYER = model.encoder.encoder.stages[-1]

    # Which disease to visualize -- defaults to whichever the model is most confident about.
    with torch.no_grad():
        _probe_out = torch.sigmoid(model((img_tensor if img_tensor.dim() == 4 else img_tensor[:, 0]).to(DEVICE))).cpu()
    GRADCAM_CLASS_IDX = int(_probe_out[0].argmax().item())  # <-- override with a specific index into `diseases` if you want a different class
    print(f"Visualizing: {diseases[GRADCAM_CLASS_IDX]} (predicted={_probe_out[0, GRADCAM_CLASS_IDX].item():.4f})")

    cam_extractor = GradCAM(model, TARGET_LAYER, RESHAPE_FN)
    _cam_input = img_tensor if img_tensor.dim() == 4 else img_tensor[:, 0]  # drop TenCrop dimension if present -- Grad-CAM needs a single crop
    cam, _ = cam_extractor(_cam_input, class_idx=GRADCAM_CLASS_IDX)

    display_img = img.resize((INPUT_SIZE, INPUT_SIZE))
    cam_resized = F.interpolate(cam.unsqueeze(0).unsqueeze(0), size=(INPUT_SIZE, INPUT_SIZE), mode="bilinear", align_corners=False)[0, 0].numpy()

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(display_img); axes[0].set_title("Original"); axes[0].axis("off")
    axes[1].imshow(cam_resized, cmap="jet"); axes[1].set_title(f"Grad-CAM: {diseases[GRADCAM_CLASS_IDX]}"); axes[1].axis("off")
    axes[2].imshow(display_img); axes[2].imshow(cam_resized, cmap="jet", alpha=0.4); axes[2].set_title("Overlay"); axes[2].axis("off")
    plt.tight_layout()
    plt.show()